# **Regression Model Training Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
group_name = "Neural Minds"
student_name = "Tahreema Rahman Zariyat"
student_id = "26016360"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [7]:
import pandas as pd
import altair as alt

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

---
## B. Business Understanding

In [9]:
business_use_case_description = """
The chosen use case is to predict the dollar value (line_total) of each line item on a sales order, given the product being sold, the special offer applied and the order quantity. A reliable model for this lets the business do the following things: Forecast expected revenue for in-flight orders before they close, flag unusual line items (actual vs. predicted) for review, estimate revenue impact of running a particular promotion on a product, support sales planning and inventory decisions.
"""

In [10]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [11]:
business_objectives = """
Accurate predictions allow finance and sales to commit to revenue forecasts with confidence and pick the right promotion mix for each product. Over-predicting line_total leads to optimistic revenue forecasts and inventory over-stocking, which ties up cash in unsold goods. Under-predicting has the opposite effect, producing pessimistic forecasts and potentially missing demand, with under-stocking and lost sales opportunities. Since the target is heavily right-skewed, errors on big-ticket lines hurt the business more than errors on cheap lines, which is why RMSE is the right thing to minimise because it is a metric that penalises large errors more heavily.
"""

In [12]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [13]:
stakeholders_expectations_explanations = """
The primary users are the finance and sales operations teams, who would consume the predicted line_total values through a dashboard or daily data extract to build revenue forecasts and flag anomalous lines for review. Marketing is a secondary user who are interested in seeing expected line value under different special offers to support promotion planning. Customers and the warehouse and supply-chain team are indirectly impacted through the better inventory and promotion decisions that flow from more accurate forecasts.
"""

In [14]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Datasets


In [15]:
try:
    df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
    print("Shape:", df.shape)
except Exception as e:
    print(e)

df.head()

Shape: (42100, 8)


,sales_order_id,sales_order_detail_id,order_quantity,product_id,special_offer_id,unit_price,unit_price_discount,line_total
0,0fbc9d71-4176-4c77-92c6-d7ede2a9ce33,354.0,1.0,6cc3d977-da73-4856-b1ca-8a2f0b57b60b,a1444fcd-4b51-4227-94cf-b13c1a2a6281,3399.99,0.0,3399.99
1,18609f9d-aedc-4665-bbc8-5d7a99599915,357.0,1.0,6cc3d977-da73-4856-b1ca-8a2f0b57b60b,a1444fcd-4b51-4227-94cf-b13c1a2a6281,3399.99,0.0,3399.99
2,0570389a-6bfa-45c2-85a4-36784fa7b5b1,359.0,1.0,41b7b226-20ea-488a-b519-b4ca65d4a64e,a1444fcd-4b51-4227-94cf-b13c1a2a6281,3578.27,0.0,3578.27
3,b48cdf2f-270a-41d0-9cde-36616fecd302,360.0,1.0,de8b1961-ea0e-4639-9482-7f71b3ed07d8,a1444fcd-4b51-4227-94cf-b13c1a2a6281,3374.99,0.0,3374.99
4,03533644-6a12-499b-9726-aaa1088c56a5,361.0,1.0,a16ed5b0-74c7-4812-adea-04b4abfd72c7,a1444fcd-4b51-4227-94cf-b13c1a2a6281,3399.99,0.0,3399.99


### C.2 Define Target variable

In [16]:
calc = df["order_quantity"] * df["unit_price"] * (1 - df["unit_price_discount"])
print("Max diff between line_total and qty*price*(1-disc):", (df["line_total"] - calc).abs().max())

Max diff between line_total and qty*price*(1-disc): 3.637978807091713e-12


In [17]:
target_definition_explanations = """
Target = line_total (dollar value of each order line). line_total = order_quantity * unit_price * (1 - discount) is an exact arithmetic identity. So if we used unit_price and discount as inputs, predicting line_total would be trivial as it is just multiplication. Here, the business problem is such that given the product, the promotion and the quantity, what dollar amount should we expect this line to bring in? So we predict line_total from product_id, special_offer_id and order_quantity. We do not use unit_price or unit_price_discount as features.
"""

In [18]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [19]:
target_name = "line_total"
y = df[target_name].copy()
print("y shape:", y.shape)

y.describe()

y shape: (42100,)


,line_total
count,42100.000000
mean,724.299768
std,1327.822757
min,1.374000
25%,9.990000
50%,49.990000
75%,939.588000
max,23667.854925


### C.4 Explore Target variable

In [20]:
alt.data_transformers.disable_max_rows()

plot_df = df[[target_name]].copy()
plot_df["log_line_total"] = np.log1p(plot_df[target_name])

h1 = alt.Chart(plot_df).mark_bar().encode(
    x=alt.X(target_name + ":Q", bin=alt.Bin(maxbins=50)),
    y="count()"
).properties(width=350, height=250, title="line_total (raw)")

h2 = alt.Chart(plot_df).mark_bar().encode(
    x=alt.X("log_line_total:Q", bin=alt.Bin(maxbins=50)),
    y="count()"
).properties(width=350, height=250, title="log(1 + line_total)")

h1 | h2

alt.HConcatChart(...)

In [21]:
target_distribution_explanations = """
line_total is heavily right-skewed with values ranging from about $1.40 to $23,668, a median around $50 and a mean around $724. A model trained to minimise MSE will naturally pay more attention to the big lines which is also what the business cares about most for revenue forecasting. I use RMSE as the headline metric with MAE and R2 as supporting metrics. I did consider log-transforming the target but decided against it, since tree-based models do not need it and keeping the raw dollar scale makes RMSE and MAE directly interpretable.
"""

In [22]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `product_id`

In [23]:
print("Number of unique products:", df["product_id"].nunique())

top_prod = df["product_id"].value_counts().head(10).reset_index()
top_prod.columns = ["product_id","lines_sold"]

print("\nTop 10 most-sold products:")
print(top_prod)

Number of unique products: 254

Top 10 most-sold products:
                             product_id  lines_sold
0  4782606a-0c9b-477e-9e0f-e996d5c298e2        2156
1  eca4e568-b2dc-4f1e-aa55-444911bfedcf        1572
2  36847649-a79a-4f70-9b5d-5c403627ab75        1457
3  c0429100-f92a-42a4-afa5-b652cee57249        1407
4  05385566-38df-4855-94a8-786cae6ffe49        1373
5  34ad3302-abb1-4d4d-9f75-b1c0c5919313        1292
6  f33d8a41-b92f-4f2a-8f75-76ea2f25722b        1232
7  5c21012d-91d4-450b-ad27-bd9311847781        1117
8  b978529c-3dff-4916-90da-9ea688df7333        1090
9  e03e45e2-c033-4832-9ece-c8bd1ff42d30         961


In [24]:
feature_1_insights = """
product_id has 254 unique values which is too many for plain one-hot encoding as it would create 254 sparse columns. The frequency distribution is uneven becasue the top products account for many more lines than the rest. My plan is to encode product_id as the mean line_total of that product in the training data. This gives the model a numeric prior about how expensive each product typically is, in a single column.
"""

In [25]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `special_offer_id`

In [26]:
print("Number of unique special offers:", df["special_offer_id"].nunique())

avg_by_offer = df.groupby("special_offer_id").agg(
    lines=("line_total","size"),
    avg_line_total=("line_total","mean")
).reset_index()

print(avg_by_offer.sort_values("avg_line_total", ascending=False))

Number of unique special offers: 10
                       special_offer_id  lines  avg_line_total
1  6468cb43-e8b4-475f-b705-f4d37f1c59eb    102     2531.788847
8  da53c22d-5867-4110-b1d0-8d84bea95ab5    148     1738.195809
9  df7c9acd-bc0c-4ab0-93ec-064e2b93af0d   1465     1317.843239
2  7a5bcdc4-f911-44d3-9567-2d0e118e94fd      6     1260.613900
6  aec729b3-e92e-4f2c-971c-4c5d426a99b9      8     1002.584250
7  c319131d-f166-40fe-ab38-cc2cb9f370d8    225      895.224610
4  a1444fcd-4b51-4227-94cf-b13c1a2a6281  40081      694.116212
0  57594a82-1a1c-497b-98dd-90e46a1c045c     33      152.033673
3  a09a7508-0e3b-42aa-942e-dde8428b1f62     20      148.558793
5  acc417ee-ae0e-4d78-8f48-3f541565c54e     12       84.531053


In [27]:
feature_2_insights = """
special_offer_id has only 10 distinct values. One offer dominates by volume because the "no discount" default offer covers approximately 98% of lines. Average line_total varies a lot across offers, so this feature does carry signal. With only 10 levels, one-hot encoding is the natural choice.
"""

In [28]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.7 Explore Feature of Interest `order_quantity`

In [29]:
q_counts = df["order_quantity"].value_counts().sort_index().reset_index()
q_counts.columns = ["order_quantity","count"]

print(q_counts.head(10))
print("\n% of lines with quantity == 1:", round((df["order_quantity"]==1).mean()*100, 1))

   order_quantity  count
0             1.0  33901
1             2.0   2352
2             3.0   1676
3             4.0   1300
4             5.0    788
5             6.0    579
6             7.0    353
7             8.0    330
8             9.0    168
9            10.0    164

% of lines with quantity == 1: 80.5


In [30]:
feature_n_insights = """
order_quantity ranges from 1 to 36. About 80% of lines are single-unit. It will be useful both as a raw numeric feature and as a "is_bulk" binary flag (1 if quantity > 1).
"""

In [31]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest `\<put feature name here\>`

> You can add more cells related to other feeatures in this section

---
## D. Feature Selection


In [32]:
features_list = ["product_id", "special_offer_id", "order_quantity"]
print("Selected features:", features_list)
print("Target          :", target_name)

Selected features: ['product_id', 'special_offer_id', 'order_quantity']
Target          : line_total


In [33]:
feature_selection_explanations = """
Three raw features are kept, that are, product_id (which product the line is for), special_offer_id (which promotion was applied) and order_quantity (how many units were sold). unit_price and unit_price_discount are dropped because together with order_quantity they directly compute line_total, so including them would turn this into arithmetic rather than prediction. sales_order_id and sales_order_detail_id are also dropped since they are just row identifiers and carry no predictive signal.
"""

In [34]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation

In [35]:
print("Before filter:", df.shape)

df = df[(df["line_total"] > 0) & (df["order_quantity"] > 0)].reset_index(drop=True)

df = df[features_list + [target_name]].copy()

print("\nAfter filter and column selection:", df.shape)
df.head()

Before filter: (42100, 8)

After filter and column selection: (42100, 4)


,product_id,special_offer_id,order_quantity,line_total
0,6cc3d977-da73-4856-b1ca-8a2f0b57b60b,a1444fcd-4b51-4227-94cf-b13c1a2a6281,1.0,3399.99
1,6cc3d977-da73-4856-b1ca-8a2f0b57b60b,a1444fcd-4b51-4227-94cf-b13c1a2a6281,1.0,3399.99
2,41b7b226-20ea-488a-b519-b4ca65d4a64e,a1444fcd-4b51-4227-94cf-b13c1a2a6281,1.0,3578.27
3,de8b1961-ea0e-4639-9482-7f71b3ed07d8,a1444fcd-4b51-4227-94cf-b13c1a2a6281,1.0,3374.99
4,a16ed5b0-74c7-4812-adea-04b4abfd72c7,a1444fcd-4b51-4227-94cf-b13c1a2a6281,1.0,3399.99


In [36]:
data_cleaning_1_explanations = """
Two cleaning steps were applied to the source data. Firstly, dropped the rows where line_total <= 0 or order_quantity <= 0. The dataset has none of these, so no rows are removed. Then, we kept only the columns we will model on.
"""

In [37]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation

In [38]:
print("Duplicates on selected columns:", df.duplicated().sum())

print("\nShape unchanged:", df.shape)

Duplicates on selected columns: 39689

Shape unchanged: (42100, 4)


In [39]:
data_cleaning_2_explanations = """
The full original dataset has no duplicate rows. Every line has a unique sales_order_detail_id). After dropping IDs, the same product/offer/quantity combination can appear many times but those are different real orders, so they stay.
"""

In [40]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation: no missing-value handling needed

In [41]:
print("Missing values in selected columns are:")
print(df.isnull().sum())

Missing values in selected columns are:
product_id          0
special_offer_id    0
order_quantity      0
line_total          0
dtype: int64


In [42]:
data_cleaning_3_explanations = """
No missing values were found in any of the selected columns, so no imputation was needed.
"""

In [43]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

---
## F. Feature Engineering

### F.1 New Feature `is_bulk`

In [44]:
df["is_bulk"] = (df["order_quantity"] > 1).astype(int)
print(df["is_bulk"].value_counts())

is_bulk
0    33901
1     8199
Name: count, dtype: int64


In [45]:
feature_engineering_1_explanations = """
is_bulk is a binary flag (1 if order_quantity > 1, else 0). It captures the main split in the data (single-unit vs multi-unit) in a single feature. It is computed from one column only so it is safe to compute before the train/test split (no risk of leakage).
"""

In [46]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature `log_order_quantity`

In [47]:
df["log_order_quantity"] = np.log1p(df["order_quantity"])
df[["order_quantity","log_order_quantity"]].describe()

,order_quantity,log_order_quantity
count,42100.000000,42100.000000
mean,1.696936,0.869241
std,2.015252,0.412381
min,1.000000,0.693147
25%,1.000000,0.693147
50%,1.000000,0.693147
75%,1.000000,0.693147
max,36.000000,3.610918


In [48]:
feature_engineering_2_explanations = """
log(1 + order_quantity) gives a smoother, less skewed version of the quantity column. It helps the linear model (Ridge) but the tree-based models are indifferent.
"""

In [49]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature `product_avg_price`

In [50]:
print("Feature engineering so far:")
print(df.columns.tolist())

Feature engineering so far:
['product_id', 'special_offer_id', 'order_quantity', 'line_total', 'is_bulk', 'log_order_quantity']


In [51]:
feature_engineering_n_explanations = """
product_id will be target-mean-encoded (mean line_total per product) but
the mean must be computed from training data only. That step is done after
the train/val/test split (i.e. section G).
"""

In [52]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [53]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

Train: (29470, 6)  Val: (6315, 6)  Test: (6315, 6)


In [54]:
data_splitting_explanations = """
Random 70 / 15 / 15 split with random_state=42 for reproducibility. The training set is used to fit models and compute the per-product mean used for target-mean encoding. The validation set is used to compare algorithms and tune hyperparameters. The test set is untouched until the very end. It is reported only once.
"""

In [55]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation: target-mean encoding for `product_id`

In [56]:
prod_avg   = train_df.groupby("product_id")["line_total"].mean()
global_avg = train_df["line_total"].mean()

for d in [train_df, val_df, test_df]:
    d["product_avg_price"] = d["product_id"].map(prod_avg).fillna(global_avg)

print("Number of products in training:", len(prod_avg))
print("\nGlobal avg used for unseen products:", round(global_avg, 2))

print()
print(train_df[["product_id","product_avg_price"]].head())

Number of products in training: 254

Global avg used for unseen products: 721.14

                             product_id  product_avg_price
0  012fbb46-4cd3-4288-b1fa-d6fe3203db0f          47.770717
1  fb4ea58b-224a-4025-ace2-f18b3c4644ab         179.952896
2  997a4288-fa52-4757-83f0-0282eff84b00        2543.713391
3  0e0dc390-a9fd-4781-b354-71c1639aec8d          35.000000
4  9014b398-fe53-4fb0-8177-97347cea83c3        2723.466561


In [57]:
data_transformation_1_explanations = """
Target-mean encoding maps each product_id to the average line_total of that product. Critically, the mean is computed from the training set only, so the validation and test sets do not see their own information during encoding. Therefore, it has no leakage. Unseen products fall back to the global training mean.
"""

In [58]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### G.3 Data Transformation: one-hot encode `special_offer_id`

In [59]:
offer_cols = pd.get_dummies(train_df["special_offer_id"], prefix="offer").columns.tolist()

def add_offer_dummies(d):
    dummies = pd.get_dummies(d["special_offer_id"], prefix="offer")
    for col in offer_cols:
        d[col] = dummies[col] if col in dummies.columns else 0
    return d

train_df = add_offer_dummies(train_df)
val_df   = add_offer_dummies(val_df)
test_df  = add_offer_dummies(test_df)

print("Added", len(offer_cols), "one-hot columns.")

Added 10 one-hot columns.


In [60]:
data_transformation_2_explanations = """
One-hot encoding is fine here because special_offer_id has only 10 levels. The column set is fixed from training so val/test have the exact same columns even if some offers are missing in those folds.
"""

In [61]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### G.4 Data Transformation: separate features from target

In [62]:
drop_cols = ["product_id", "special_offer_id", "line_total"]

X_train = train_df.drop(columns=drop_cols)
X_val   = val_df.drop(columns=drop_cols)
X_test  = test_df.drop(columns=drop_cols)

y_train = train_df["line_total"].copy()
y_val   = val_df["line_total"].copy()
y_test  = test_df["line_total"].copy()

print("X_train columns:")
print(X_train.columns.tolist())

print("\nShapes -> X_train:", X_train.shape, " y_train:", y_train.shape)

X_train columns:
['order_quantity', 'is_bulk', 'log_order_quantity', 'product_avg_price', 'offer_57594a82-1a1c-497b-98dd-90e46a1c045c', 'offer_6468cb43-e8b4-475f-b705-f4d37f1c59eb', 'offer_7a5bcdc4-f911-44d3-9567-2d0e118e94fd', 'offer_a09a7508-0e3b-42aa-942e-dde8428b1f62', 'offer_a1444fcd-4b51-4227-94cf-b13c1a2a6281', 'offer_acc417ee-ae0e-4d78-8f48-3f541565c54e', 'offer_aec729b3-e92e-4f2c-971c-4c5d426a99b9', 'offer_c319131d-f166-40fe-ab38-cc2cb9f370d8', 'offer_da53c22d-5867-4110-b1d0-8d84bea95ab5', 'offer_df7c9acd-bc0c-4ab0-93ec-064e2b93af0d']

Shapes -> X_train: (29470, 14)  y_train: (29470,)


In [63]:
data_transformation_3_explanations = """
Final feature matrix X has only numeric columns and the target y is held separately. X_train, X_val and X_test all have the same columns in the same order, which is required for sklearn estimators.
"""

In [64]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## H. Save Datasets

> Do not change this code

In [65]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

## J. Train Machine Learning Model

### J.1 Import Algorithm

> Provide some explanations on why you believe this algorithm is a good fit


In [66]:
algorithms = {
    "Ridge": Ridge(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

print("Algorithms to try:", list(algorithms.keys()))

Algorithms to try: ['Ridge', 'RandomForest', 'GradientBoosting']


In [67]:
algorithm_selection_explanations = """
I picked three algorithms from different families so that I can compare them. Firstly. Ridge is a linear model with L2 regularisation. It is cheap, interpretable and useful as a floor to see how much of the signal is actually linear. RandomForest is a bagging-based tree ensemble that handles non-linear relationships and feature interactions automatically and is robust to outliers. The second algorithm, GradientBoosting is a boosted tree ensemble that is usually the strongest of the three for tabular regression problems. I expect the tree-based models to win here because product_avg_price has a non-linear relationship with line_total and the multiplicative structure around order_quantity is best captured by feature interactions which trees handle natively while a linear model cannot.
"""

In [68]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

> Provide some explanations on why you believe this algorithm is a good fit


In [69]:
param_grids = {
    "Ridge": {
        "alpha": [0.1, 1.0, 10.0]
    },
    "RandomForest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20]
    },
    "GradientBoosting": {
        "n_estimators": [100, 200],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1]
    },
}
print("Hyperparameter grids defined.")

Hyperparameter grids defined.


In [70]:
hyperparameters_selection_explanations = """
I am keeping the grids small to stay fast. For Ridge, I tune alpha which controls regularisation strength (0.1 is light, 10 is strong). For RandomForest, I tune n_estimators and max_depth since more trees and deeper trees usually reduce variance but cost more time. For GradientBoosting, I tune n_estimators, max_depth, and learning_rate, the classic trio for boosting, a lower learning_rate needs more trees but generalises better, while max_depth controls how complex each individual tree is. These small grids are enough to demonstrate the tuning workflow without taking too long to run.
"""

In [71]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [72]:
best_models = {}

for name, model in algorithms.items():
    print(f"\nTuning {name} ...")
    gs = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring="neg_root_mean_squared_error",
        cv=3,
        n_jobs=-1
    )
    gs.fit(X_train, y_train)
    best_models[name] = gs.best_estimator_
    print(f"  Best params: {gs.best_params_}")
    print(f"  Best CV RMSE: ${-gs.best_score_:.2f}")


Tuning Ridge ...
  Best params: {'alpha': 1.0}
  Best CV RMSE: $674.28

Tuning RandomForest ...
  Best params: {'max_depth': None, 'n_estimators': 200}
  Best CV RMSE: $141.94

Tuning GradientBoosting ...
  Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
  Best CV RMSE: $120.55


### J.4 Model Technical Performance

> Provide some explanations on model performance


In [73]:
def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f" {name:18s} RMSE=${rmse:>10,.2f} MAE=${mae:>9,.2f} R2={r2:>7.4f}")

for name, model in best_models.items():
    print(f"\n{name}")
    report("Train", y_train, model.predict(X_train))
    report("Val", y_val, model.predict(X_val))
    report("Test", y_test, model.predict(X_test))

best_name = min(best_models,
                key=lambda n: np.sqrt(mean_squared_error(y_val, best_models[n].predict(X_val))))

print(f"\nBest model on validation: {best_name}")


Ridge
 Train              RMSE=$    673.17 MAE=$   290.00 R2= 0.7436
 Val                RMSE=$    673.83 MAE=$   285.11 R2= 0.7409
 Test               RMSE=$    659.13 MAE=$   290.37 R2= 0.7521

RandomForest
 Train              RMSE=$    103.66 MAE=$    34.65 R2= 0.9939
 Val                RMSE=$    136.00 MAE=$    40.67 R2= 0.9894
 Test               RMSE=$    142.13 MAE=$    42.97 R2= 0.9885

GradientBoosting
 Train              RMSE=$    106.99 MAE=$    43.19 R2= 0.9935
 Val                RMSE=$    120.62 MAE=$    45.64 R2= 0.9917
 Test               RMSE=$    126.64 MAE=$    48.51 R2= 0.9908

Best model on validation: GradientBoosting


In [74]:
model_performance_explanations = """
After tuning on the validation set (random_state=42), Ridge produced RMSE ~$674, MAE ~$285 and R2 ~0.74, RandomForest produced RMSE ~$136, MAE ~$41 and R2 ~0.99, and GradientBoosting produced RMSE ~$121, MAE ~$46 and R2 ~0.99. Compared to the mean-predictor baseline (RMSE ~$1,323), Ridge already cuts validation RMSE roughly in half, which shows there is useful linear signal in product_avg_price and the offer dummies. Both tree-based models do dramatically better with R2 around 0.99. That means, they explain almost all of the variance in line_total. Tuned GradientBoosting wins on RMSE which is the metric set out to be minimised, while RandomForest is slightly ahead on MAE, but the two are very close. Test-set numbers track validation numbers closely, so there is no obvious overfitting. The reason the tree models perform so strongly is that product_avg_price encodes the typical dollar value per product, and tree models can learn the rule "line_total ~ order_quantity * product_avg_price" with just a few splits, even without unit_price itself in the feature set.
"""

In [75]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

> Provide some analysis on the model impacts from the business point of view


In [76]:
best_model = best_models[best_name]
test_pred = best_model.predict(X_test)
errors = test_pred - y_test.values

print("Test-set error summary for the best model ({}):".format(best_name))
print(f" Mean error (bias) : ${errors.mean():.2f}")
print(f" Median absolute error : ${np.median(np.abs(errors)):.2f}")
print(f" 90th pct absolute error : ${np.percentile(np.abs(errors),90):.2f}")
print(f" 99th pct absolute error : ${np.percentile(np.abs(errors),99):.2f}")
print(f" Worst absolute error : ${np.abs(errors).max():.2f}")

Test-set error summary for the best model (GradientBoosting):
 Mean error (bias) : $-2.08
 Median absolute error : $3.47
 90th pct absolute error : $136.50
 99th pct absolute error : $695.73
 Worst absolute error : $2561.42


In [77]:
business_impacts_explanations = """
Translating model performance into business terms, the mean absolute error on test is roughly $40 to $50, compared to about $900 for the baseline mean-predictor (a ~95% reduction in typical error). A finance team forecasting daily revenue from this model would be off by a few tens of dollars per line on average, versus hundreds with no model. The errors that remain are concentrated in the tail are the 99th percentile absolute error is much larger than the median which means a small number of unusual lines (very large bulk orders or rare products) drive most of the residual risk. Over-prediction biases would inflate revenue forecasts and risk over-stocking, while under-prediction risks under-supplying popular items. My recommendation is to deploy the chosen model as an "expected line value" service and pair its output with a confidence band, so the finance team can flag rows where the predicted value is far from any historical norm for that product.
"""

In [78]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## H. Project Outcomes

In [79]:
experiment_outcome = "Hypothesis Confirmed"

In [80]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [81]:
experiment_results_explanations = """
The outcome is that the hypothesis is confirmed. The hypothesis was that line_total can be predicted from product, offer and quantity alone without unit_price as a feature by giving the model a target-mean-encoded view of each product. The tree-based models confirm this with validation and test R2 around 0.99 and MAE around $40 to $50, approximately 95% reduction versus the mean-predictor baseline. The main insight is that most of the predictive power comes from product_avg_price, Once the model knows the typical dollar value for a product, it only needs to adjust for quantity and offer. Linear models like Ridge leave a lot on the table here (RMSE ~$680 vs ~$140 for trees), because the product-by-quantity multiplication is non-linear and trees handle that with no extra work, whereas a linear model cannot. It can also be noticed that order_quantity is highly imbalanced (about 80% single-unit), so adding more multi-unit examples or better features for bulk lines would likely help the tail. As next steps ranked by expected uplift, the highest impact change would be joining sales_order_header to bring in order date and customer features, which would unlock time-aware splits, seasonality and customer-level encoding. Adding product_category and sub_category as features once those tables are joined would help with cold-start on unseen products. Trying XGBoost or LightGBM with early stopping would likely give a small but nice uplift over the sklearn boosters. For production, the target-mean encoding should be wrapped inside a sklearn Pipeline so the deployed model handles encoding for new rows automatically with data-drift monitoring on product_avg_price values.
"""

In [82]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)